# 16.1 The Type System — What a Checker Actually Does

**Prerequisites:** 4.5 Type Hints for Functions, 15 Testing and Debugging  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 Annotations are **data, not enforcement** — Python does not check them
- What a static checker buys you, and what it costs
- `reveal_type()` — the tool you will use more than any other
- 🔴 **`Any` is not a type, it is an off switch** — and it spreads
- Gradual typing: how an unannotated function silently disables checking
- The `mypy` command line, error codes, and `--strict`
- `# type: ignore` — when it is right, and 🔴 two ways to write it wrong
- 🔴 What type checking does **not** catch (compare **15.6** on coverage)

---

## Where this starts

**4.5** taught the *syntax*: `def add(a: int, b: int) -> int`, `Optional`, `Callable`,
`Sequence`. This folder is about the **system** behind that syntax — what a checker can prove,
what it cannot, and how to use it on real code.

The one thing to understand before anything else:

> **Python does not check annotations. Ever.**
> They are stored as data and otherwise ignored. All the value comes from a *separate program*
> — `mypy`, `pyright`, `ty` — reading your source and reasoning about it.

That separation is deliberate, and it is why type hints could be added to a 30-year-old dynamic
language without breaking anything. It is also the source of every misunderstanding about them.

Everything below runs **`mypy` for real** in a temporary directory, so you see actual checker
output rather than a description of it.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py161_"))


def write(name, source):
    path = WORK / name
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


def mypy(name, source=None, *flags, show_source=False):
    """Type-check a file with mypy and return its report."""
    if source is not None:
        write(name, source)
    done = subprocess.run(
        [sys.executable, "-m", "mypy", name,
         "--cache-dir", str(WORK / ".mypy_cache"),   # keep the cache out of the repo
         "--no-color-output", "--no-error-summary", *flags],
        cwd=WORK, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    report = (done.stdout + done.stderr).strip() or "(mypy found nothing to report)"
    header = f"$ mypy {name} {' '.join(flags)}".rstrip()
    return f"{header}\n{'-' * 68}\n{report}\n{'-' * 68}\nexit code: {done.returncode}"


def python(name):
    """Actually run the file, to contrast with what mypy said."""
    done = subprocess.run([sys.executable, name], cwd=WORK, capture_output=True,
                          text=True, encoding="utf-8", errors="replace", timeout=60)
    report = (done.stdout + done.stderr).strip()
    return (f"$ python {name}\n{'-' * 68}\n{report}\n"
            f"{'-' * 68}\nexit code: {done.returncode}")


print("scratch:", WORK)
print("mypy   :", subprocess.run([sys.executable, "-m", "mypy", "--version"],
                                 capture_output=True, text=True).stdout.strip())

## 🔴 Annotations are data, not enforcement

The next cell type-checks a file, then **runs the same file**. Read both outputs.

In [ ]:
NO_ENFORCEMENT = r"""
    def cache_key(user_id: int, region: str) -> str:
        return f"user:{user_id}:{region}"


    # Both arguments are the wrong type, and the wrong way round.
    print(cache_key("not-an-int", 42))

    # The annotations are just a dict hanging off the function object:
    print(cache_key.__annotations__)
"""

print(mypy("noenforce.py", NO_ENFORCEMENT))
print()
print(python("noenforce.py"))

mypy found two errors. Python ran it happily and printed
`user:not-an-int:42`.

🔴 **Nothing at runtime consulted those annotations.** They were stored in
`cache_key.__annotations__` and ignored. If you want runtime enforcement you need a library
that reads the annotations and acts on them — `pydantic` is the usual choice, and it belongs to
**18 Working with APIs**, where untrusted input actually arrives.

### So what is the point?

| You get | Because |
|---|---|
| Bugs found without running the code | the checker reasons about *every* path, not the ones your tests happen to hit |
| Refactoring safety | rename a field and the checker lists every caller |
| Editor autocomplete that is actually right | the same information powers your IDE |
| Documentation that cannot rot | a wrong annotation is a *reported error*, unlike a comment |

| It costs | |
|---|---|
| Annotation effort | worst on highly dynamic code |
| A second tool in CI | seconds, but it has to be wired up (**15.6**) |
| Learning curve | generics and variance (**16.3**) are genuinely hard |

The honest summary: **types are a second opinion, not a proof.** They are strongest exactly
where tests are weakest — the paths nobody thought to test — and weakest where tests are
strongest, which is checking that the answer is *right*.

## `reveal_type()` — the tool you will use constantly

You cannot debug a type error without knowing what the checker thinks a value *is*.
`reveal_type(x)` makes it say so.

🔴 There are **two different things** with this name:

| Form | Nature | At runtime |
|---|---|---|
| bare `reveal_type(x)` | a **mypy pseudo-function** — no import | 💥 `NameError` |
| `from typing import reveal_type` | a real function (3.11+) | prints, and returns `x` |

The bare form is the one you will type. It is a *debugging aid you delete*, exactly like a
`print` (**15.7**) — and if you forget to delete it, your program crashes.

In [ ]:
print(mypy("reveal.py", r"""
    counts: dict[str, int] = {"queued": 3, "running": 1}

    reveal_type(counts)
    reveal_type(counts["queued"])
    reveal_type(counts.get("done"))          # .get can miss - note the None
"""))

Three answers, and the third is the useful one: `counts.get("done")` is
**`int | None`**, not `int`, because `.get` returns `None` for a missing key (**2.5**). That is
the single most common source of type errors in real code, and `reveal_type` is how you see it.

Now the runtime side. The **bare** form crashes; the imported one works and returns its
argument, so you can wrap an expression in it without changing behaviour.

In [ ]:
print(python(str(write("bare.py", r"""
    counts = {"queued": 3}
    reveal_type(counts)              # no import - mypy-only
""").name)))
print()
print(python(str(write("imported.py", r"""
    from typing import reveal_type

    counts = {"queued": 3}
    same = reveal_type(counts)       # prints AND returns its argument
    print("returned:", same)
""").name)))

`NameError` for the bare form; `Runtime type is 'dict'` for the imported one.
Treat the bare form like a breakpoint (**15.8**): useful, temporary, and never committed.

## 🔴 `Any` is an off switch, not a type

`Any` means *"stop checking this"*. It is compatible with everything in both directions, so a
single `Any` can silently disable checking across a whole call chain.

The most common way it appears is not by writing `Any` at all — it is by **leaving a return
annotation off**.

In [ ]:
print(mypy("anyleak.py", r"""
    from collections.abc import Sequence


    def first_untyped(items: Sequence[int]):            # 🔴 no return annotation
        return items[0] if items else None


    def first_typed(items: Sequence[int]) -> int | None:
        return items[0] if items else None


    reveal_type(first_untyped([1, 2, 3]))
    reveal_type(first_typed([1, 2, 3]))

    a = first_untyped([1])
    a.completely_made_up_method()          # no error at all

    b = first_typed([1])
    b.completely_made_up_method()          # 🔴 caught
"""))

Two functions with **identical bodies**. The unannotated one returns `Any`,
so `a.completely_made_up_method()` passed without complaint. The annotated one returned
`int | None` and the checker caught both problems.

That is gradual typing working as designed — unannotated code is *assumed fine* so you can
adopt types incrementally (**16.6**) — but it means **the parts you have not annotated are not
merely unchecked; they can hide errors in the parts you have.**

### `Any` vs `object`

They look similar and are opposites. This is the single most useful distinction in the folder.

In [ ]:
print(mypy("anyobject.py", r"""
    from typing import Any


    def takes_any(value: Any) -> None:
        value.anything()           # allowed: Any permits every operation
        print(value + 1)


    def takes_object(value: object) -> None:
        value.anything()           # 🔴 object permits almost nothing
        print(value + 1)


    takes_any("a string")          # any argument is accepted
    takes_object("a string")       # so is this - object is the top type
"""))

| | `Any` | `object` |
|---|---|---|
| What can be **passed in** | anything | anything |
| What you can **do with it** | 🔴 **everything** — unchecked | almost nothing until you narrow it |
| Meaning | "I have given up" | "I genuinely do not care what this is" |

🔴 **`object` is the honest choice** when a function truly accepts anything — a logging helper,
a serialiser. It forces you to narrow with `isinstance` before using the value, which is
exactly right. `Any` should be a deliberate, commented escape hatch.

## Running the checker

```bash
mypy src/                      # check a package
mypy --strict src/             # every optional check on
mypy --check-untyped-defs .    # look inside unannotated functions too
mypy --show-error-codes .      # on by default in modern mypy
```

Every error carries a **code** in brackets — `[arg-type]`, `[return-value]`, `[union-attr]`,
`[no-untyped-def]`. Those codes matter: they are what you silence selectively, and what you
search for when the message is unfamiliar.

Here is the same unannotated file under the default settings and under `--strict`.

In [ ]:
LOOSE = r"""
    def load_budget(raw):
        return int(raw)


    def total(rows):
        return sum(load_budget(row) for row in rows)


    print(total(["1", "2", "3"]))
"""

print(mypy("loose.py", LOOSE))
print()
print(mypy("loose.py", None, "--strict"))

Default mypy reported **nothing** — the file is entirely unannotated, so
there is nothing to contradict. `--strict` reported three problems, because it insists that
functions *be* annotated.

That contrast is the whole adoption story: **mypy's default posture is "check what you have
told me"**, and strictness is how you raise the bar as coverage grows (**16.6**).

### The flags worth knowing

`--strict` is a bundle. These are the members you will meet by name:

| Flag | Catches |
|---|---|
| `--disallow-untyped-defs` | functions with no annotations |
| `--disallow-any-generics` | bare `list` instead of `list[str]` |
| `--warn-return-any` | returning `Any` from a function declared otherwise |
| `--warn-unused-ignores` | 🔴 `type: ignore` comments no longer needed |
| `--warn-unreachable` | code the checker can prove never runs |
| `--no-implicit-reexport` | imports leaking out of your module as public API |
| `--check-untyped-defs` | 🔴 *not* in `--strict` — looks **inside** unannotated bodies |

`--check-untyped-defs` is the best first move on an untyped codebase: it finds real bugs
without requiring you to annotate anything.

## `# type: ignore`

Sometimes the checker is wrong, or right but unhelpful — an untyped dependency, a genuinely
dynamic pattern. `# type: ignore` silences a line.

```python
value = legacy_api.fetch()  # type: ignore[no-any-return]
```

🔴 **Always give the code in brackets.** A bare `# type: ignore` hides *every* error on that
line, including ones that appear later for completely different reasons.

In [ ]:
print(mypy("ignores.py", r"""
    def needs_the_ignore(x: int) -> str:
        return x  # type: ignore[return-value]


    def does_not_need_it(x: int) -> int:
        return x  # type: ignore[return-value]
""", "--warn-unused-ignores"))

`--warn-unused-ignores` found the stale one. Without that flag, `type: ignore`
comments accumulate for years and quietly hide real errors long after the original reason is
gone — the type-checking equivalent of a skipped test (**15.2**).

### 🔴 The formatting trap

The bracketed form is parsed strictly. Anything after the closing bracket makes the **whole
comment invalid** — and mypy then reports the error you were trying to silence, plus a second
one about the comment.

In [ ]:
print(mypy("badignore.py", r"""
    def f(x: int) -> str:
        return x  # type: ignore[return-value]  because the API is untyped
"""))
print()
print("Write the prose FIRST, or use a separate line:")
print(mypy("goodignore.py", r"""
    def f(x: int) -> str:
        # the vendor API is untyped and returns str at runtime
        return x  # type: ignore[return-value]
"""))

Two errors from one badly-formatted comment. The fix is to put the
explanation on its **own line above**.

## 🔴 What type checking does *not* catch

This is the section to remember, and it mirrors **15.6**'s point about coverage: a green check
is a **lower bound** on correctness, not an upper bound.

The next file is **fully annotated** and passes `--strict` cleanly. It also contains three real
bugs.

In [ ]:
BUGGY = r"""
    from collections.abc import Sequence


    def retry_delay(attempt: int, base: float = 1.0, ceiling: float = 30.0) -> float:
        delay = base
        for _ in range(attempt):
            delay = delay * 2
        return max(delay, ceiling)                    # BUG 1: max, should be min


    def average(values: Sequence[float]) -> float:
        return sum(values) / len(values)              # BUG 2: empty -> ZeroDivisionError


    def is_retryable(status: int, attempts: int) -> bool:
        return status in (500, 502, 503) and attempts < 3   # BUG 3: 429 missing


    print("retry_delay(1) =", retry_delay(1), " (should be 2.0)")
    print("is_retryable(429, 0) =", is_retryable(429, 0), " (should be True)")
"""

print(mypy("buggy.py", BUGGY, "--strict"))
print()
print(python("buggy.py"))

**Clean under `--strict`. Three bugs.** And they are not exotic:

| Bug | Why the checker cannot see it |
|---|---|
| `max` where `min` was meant | both return `float`; the **types are right, the logic is wrong** |
| division by zero on an empty sequence | `len()` returning `0` is a perfectly valid `int` |
| `429` missing from the retryable set | 🔴 **missing code has no type** — exactly the point **15.6** made about coverage |

> **Types and tests are complementary, and neither subsumes the other.**
>
> | | Types catch | Tests catch |
> |---|---|---|
> | Wrong *kind* of value | ✅ every call site | only the paths you exercise |
> | Wrong *logic* | ❌ never | ✅ that is their whole job |
> | Missing requirement | ❌ | ✅ if you wrote the test |
> | `None` on an unexercised path | ✅ | ❌ unless the test hits it |
>
> A codebase with types and no tests is confidently wrong. One with tests and no types has to
> discover `None` bugs at runtime, one path at a time.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Expecting annotations to be enforced.** Python ignores them entirely. Runtime validation needs a library like `pydantic` (**18**).
2. 🔴 **Leaving the return annotation off.** The function returns `Any`, and `Any` silences every check on the value downstream.
3. 🔴 **A bare `# type: ignore`.** It hides every error on that line, forever. Always write `# type: ignore[error-code]`.
4. **Prose after `# type: ignore[code]`.** It makes the comment invalid and you get two errors instead of none. Put the explanation on the line above.
5. **Never running `--warn-unused-ignores`.** Stale ignores accumulate and hide real errors long after their reason has gone.
6. **Reaching for `Any` when you mean `object`.** `Any` disables checking; `object` says “I do not care what this is” and still makes you narrow it.
7. **Committing a bare `reveal_type()`.** It is a mypy pseudo-function — at runtime it is a `NameError`.
8. **Believing a green check means correct.** Types cannot see wrong logic, and cannot see code you never wrote.
9. **Starting a legacy codebase on `--strict`.** You get thousands of errors and give up. Start with `--check-untyped-defs` (**16.6**).

## Best Practices

- Annotate **return types** first — that is where `Any` leaks in.
- Use `reveal_type()` the moment an error message confuses you; delete it immediately after.
- Prefer `object` to `Any` when a value really can be anything.
- Always write the error code in a `type: ignore`, with the reason on the line above.
- Turn on `--warn-unused-ignores` so silenced errors cannot outlive their reason.
- Run `--check-untyped-defs` on any codebase before annotating anything — free bug-finding.
- Treat the checker as a second opinion alongside tests, not a replacement for them.
- Put mypy in CI next to pytest (**15.6**), so type errors fail the build like test failures.

## Practice Exercises

Try these before moving on.

1. Write a function with correct annotations, call it with the wrong types, and confirm mypy objects while Python runs it. Then print its `__annotations__`.
2. 🔴 Take a function of your own, remove its return annotation, and use `reveal_type` to watch a caller's variable become `Any`. How far downstream does the damage reach?
3. Rewrite `takes_any` from this notebook with `object` and make it type-check by narrowing with `isinstance`. Which version would you rather maintain?
4. Run `mypy --strict` on any notebook's code from **04 Functions**. Fix the errors. How many were real bugs rather than missing annotations?
5. Add `# type: ignore` to a line, then delete the code that made it necessary, and prove `--warn-unused-ignores` catches it.
6. 🔴 Take the three-bug file from this notebook and write pytest tests (**15.1**) that catch all three. What does that tell you about where each tool earns its place?
7. **Interview question:** a colleague says “we have full type coverage, so we need fewer tests”. What do you say?

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | 🔴 **PEP 695** — `def first[T](...)` and the `type` statement, a completely new generics syntax (**16.3**) |
| **3.11** | `typing.reveal_type` added as a real, importable function; `Self` (**16.4**); `assert_never` (**16.2**) |
| **3.10** | 🔴 `X | Y` unions, replacing `Union[X, Y]` and `Optional[X]`; `ParamSpec` (**16.3**); `TypeGuard` |
| **3.9** | `list[str]` / `dict[str, int]` builtins, replacing `List` / `Dict` from `typing` |
| **3.8** | `Protocol`, `TypedDict`, `Literal`, `Final` — the release that made typing genuinely usable |

Written against **mypy 2.3.1** on **Python 3.14.4**.

## Where next

| Notebook | Covers |
|---|---|
| **16.2** | unions, narrowing, `Literal`, `Final`, `Annotated`, `TypedDict` |
| **16.3** | generics — `TypeVar`, PEP 695 syntax, variance, `ParamSpec` |
| **16.4** | protocols and structural typing, `Self`, `@overload` |
| **16.5** | typing real code: classes, decorators, async, third-party stubs |
| **16.6** | adopting types in an existing codebase |

## Related

- **4.5 Type Hints for Functions** — the syntax this folder builds on
- **15.6 Testing in Practice** — the same "green does not mean correct" lesson, for coverage
- **15.7** — `print` vs logging vs debugger; `reveal_type` is the type-level equivalent
- **18 Working with APIs** — `pydantic`, for when you need runtime enforcement